In [ ]:
# fix imports
import os
import sys

module_path = os.path.abspath(os.path.join(".."))
if module_path not in sys.path:
    sys.path.append(module_path)

In [ ]:
# import torch
# from notebooks.experiment_robust import load_robust_experiment
# from notebooks.experiment_torch import load_torchvision_experiment


# # model, dl_train, dl_eval = load_robust_experiment("Salman2020Do_R50", "imagenet", batch_size=256)
# # model, dl_train, dl_eval = load_robust_experiment("Singh2023Revisiting_ViT-S-ConvStem", "imagenet", batch_size=256)
# # model, dl_train, dl_eval = load_robust_experiment("Standard_R50", "imagenet", batch_size=256)
# model, dl_train, dl_eval = load_torchvision_experiment("wong-resnet50", batch_size=256)

# dl_eval.batch_size = 256

In [ ]:
from scripts.utils.load_model import load_model
from scripts.utils.load_dataset import load_dataset

model, trans = load_model("wong-resnet50", dataset_name="imagenet")
dl_train, dl_eval, dl_test = load_dataset("imagenet", trans, train_batch=256, eval_batch=256)

In [ ]:
model

In [ ]:
import importlib
# reload ActivationExtractor to ensure we have the latest version
import ulib.utils.activation_extractor
importlib.reload(ulib.utils.activation_extractor)
from ulib import StopCriteria, PertModule, ActivationExtractor

In [ ]:
from ulib import StopCriteria, PertModule, ActivationExtractor
from ulib.attacks.fg_uap import FG_UAP
import torch

stop = StopCriteria(max_epochs=20, max_time=20 * 60)
pert_model = PertModule(model, data_shape=dl_train.get_tensor(0).shape[1:], eps=16 / 255, random_init=True, input_clamp=True)
optimizer = torch.optim.Adam(pert_model.parameters(), lr=0.02)
extractor = ActivationExtractor(model, "model.fc", exact_match=True, capture_output=False)


x, y = next(iter(dl_eval))

with extractor.capture():
    out = pert_model(x.to(pert_model.device))
    
print(out.shape)
print(extractor.get_activations())

In [ ]:
# check if there is a gradient w.r.d the perturbation
# out.mean().backward()
print(pert_model.get_pert(clone=False).grad)

In [ ]:
aaa

In [ ]:
from ulib import StopCriteria, PertModule, ActivationExtractor
from ulib.attacks.fg_uap import FG_UAP
import torch

stop = StopCriteria(max_epochs=20, max_time=20 * 60)
pert_model = PertModule(model, data_shape=dl_train.get_tensor(0).shape[1:], eps=16 / 255, random_init=True, input_clamp=True)
optimizer = torch.optim.Adam(pert_model.parameters(), lr=0.02)
extractor = ActivationExtractor(model, "model.fc", exact_match=True, capture_output=False)


attack = FG_UAP(
    pert_model=pert_model,
    optimizer=optimizer,
    activ_extractor=extractor,
    # scheduler=scheduler,
    mixed_precision=False,
)

pert = attack.fit(dl_train, dl_eval, stop)

In [ ]:
from ulib.evaluator import ExtendedEvaluator
from ulib.utils.visualize import display_pert

evaluator = ExtendedEvaluator(pert_model, verbose=True)
print(evaluator.evaluate(dl_eval))
ax = display_pert(pert_model)

In [ ]:
ax = display_pert(pert_model)
# ax.get_figure().savefig("r50-fg-uap.pdf", format="pdf", bbox_inches="tight")